# Explainable AI for the selected Extended CNNs

Notebook 03 ranks all exported Extended-CNN candidates on the fixed `validate` fold and writes an ordered top-three manifest **before loading test**. This notebook consumes exactly that frozen manifest: it does not discover models independently and does not change their ranking. Explanations use `validate` trajectories only; test remains a final performance estimate rather than an explanation-development set.

Three complementary views are used: Layer Grad-CAM localizes influential convolutional activations, Integrated Gradients attributes the prediction back to input time points, and Temporal Occlusion measures the target-logit change after masking windows. Agreement is useful, but visual agreement alone is not treated as biological proof.

## Contents

| Section | Question |
|---|---|
| [1. Frozen selection and data](#setup) | Which models and samples are explained? |
| [2. Attribution protocol](#protocol) | What do the three methods measure? |
| [3. Per-dose attribution maps](#maps) | Which time regions matter by dose? |
| [4. Representative trajectories](#examples) | How do explanations align with actual curves? |
| [5. Spectral Grad-CAM](#spectral) | Does an FFT branch emphasize particular frequencies? |
| [6. Agreement and deletion tests](#validation) | Are maps stable across methods and functionally relevant? |
| [7. Interpretation checklist](#interpretation) | Which claims are supported and which remain hypotheses? |

<a id="setup"></a>
## 1. Frozen selection and data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr

from pytorch_timecourse_classification.artifacts import (
    artifact_root, load_model, load_selection_manifest,
)
from pytorch_timecourse_classification.data import (
    Preprocessor, load_prepared_fold, preprocessors_compatible,
)
from pytorch_timecourse_classification.explainability import (
    explainable_layers, integrated_gradients, layer_gradcam,
    normalize_attributions, temporal_occlusion, temporal_window_deletion,
)

In [ ]:
manifest = load_selection_manifest("extended_cnns_top3.json")
if manifest["selection_fold"] != "validate" or manifest["selection_metric"] != "macro_f1":
    raise ValueError("Notebook 04 expects a validate/macro-F1 selection from Notebook 03.")
selected = manifest["candidates"]
if len(selected) != 3:
    raise ValueError(f"Expected exactly three selected architectures, got {len(selected)}.")
experiment_names = tuple(candidate["experiment"] for candidate in selected)
print("Frozen tune-only selection:")
display(pd.DataFrame(selected).set_index("rank").round(4))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models, checkpoints = {}, {}
canonical_preprocessor = None
for experiment_name in experiment_names:
    model, checkpoint = load_model(artifact_root() / experiment_name / "model.pt", device=device)
    if getattr(model, "model_name", None) != "extended_cnn":
        raise TypeError(f"{experiment_name} is not an Extended CNN artifact.")
    preprocessor = Preprocessor(
        tuple(checkpoint["class_names"]), float(checkpoint["signal_mean"]),
        float(checkpoint["signal_std"]),
    )
    if canonical_preprocessor is None: canonical_preprocessor = preprocessor
    elif not preprocessors_compatible(canonical_preprocessor, preprocessor):
        raise ValueError("Selected models use incompatible preprocessing.")
    models[experiment_name] = model
    checkpoints[experiment_name] = checkpoint
tune_fold = load_prepared_fold("validate", canonical_preprocessor)
class_names = canonical_preprocessor.class_names
print(f"device: {device}; tune shape: {tuple(tune_fold.features.shape)}")

In [ ]:
SAMPLES_PER_DOSE = 12
rng = np.random.default_rng(42)
targets_numpy = tune_fold.targets.numpy()
analysis_indices = np.concatenate([
    rng.choice(np.flatnonzero(targets_numpy == class_index), size=min(SAMPLES_PER_DOSE, np.sum(targets_numpy == class_index)), replace=False)
    for class_index in range(len(class_names))
])
analysis_indices.sort()
analysis_inputs = tune_fold.features[analysis_indices].to(device)
analysis_targets = tune_fold.targets[analysis_indices].to(device)
print("Balanced explanation cohort:", dict(zip(class_names, np.bincount(analysis_targets.cpu().numpy()))))

<a id="protocol"></a>
## 2. Attribution protocol

All methods explain the logit of the **true dose class**, not whichever class the model happened to predict. This prevents incorrect predictions from silently receiving a different explanation target. Zero is used as the masking and integration baseline because the inputs were globally z-standardized using train statistics; it corresponds to the training mean signal. Baseline sensitivity remains a limitation and should later be checked against mean and smoothed trajectory baselines.

- **Integrated Gradients:** signed accumulated input sensitivity from the zero baseline to the observed trajectory.
- **Temporal Occlusion:** target-logit drop after replacing overlapping windows by the baseline.
- **Layer Grad-CAM:** coarse positive class evidence from the final convolution of each temporal branch, interpolated back to input length. FFT-branch CAMs remain in frequency space and are never presented as temporal maps.

In [ ]:
TIME_LENGTH = analysis_inputs.shape[-1]
FFT_LENGTH = TIME_LENGTH // 2 + 1
attributions, spectral_cams = {}, {}
for experiment_name, model in models.items():
    ig = integrated_gradients(model, analysis_inputs, analysis_targets, steps=64)
    occlusion = temporal_occlusion(
        model, analysis_inputs, analysis_targets, window_size=17, stride=4,
    )
    temporal_maps, frequency_maps = [], {}
    for layer in explainable_layers(model):
        output_length = TIME_LENGTH if layer.domain == "time" else FFT_LENGTH
        cam = layer_gradcam(
            model, analysis_inputs, analysis_targets, layer, output_length=output_length,
        )
        if layer.domain == "time": temporal_maps.append(normalize_attributions(cam))
        else: frequency_maps[layer.name] = normalize_attributions(cam)
    gradcam = torch.stack(temporal_maps).mean(dim=0)
    attributions[experiment_name] = {
        "Integrated Gradients": normalize_attributions(ig),
        "Temporal Occlusion": normalize_attributions(occlusion),
        "Layer Grad-CAM": normalize_attributions(gradcam),
    }
    spectral_cams[experiment_name] = frequency_maps
    print(experiment_name, getattr(model, "architecture", None), [layer.name for layer in explainable_layers(model)])

<a id="maps"></a>
## 3. Per-dose attribution maps

Each row averages independently normalized sample maps for one true dose. This reveals recurring temporal preferences without letting a few high-amplitude attribution maps dominate the mean. It does not show uncertainty, so the later sample-level agreement and deletion analyses remain necessary.

In [ ]:
time_axis = np.arange(TIME_LENGTH)
for experiment_name, methods in attributions.items():
    figure, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True, sharey=True)
    for axis, (method_name, values) in zip(axes, methods.items()):
        dose_maps = torch.stack([
            values[analysis_targets == class_index].mean(dim=0).squeeze(0)
            for class_index in range(len(class_names))
]).cpu().numpy()
        image = axis.imshow(dose_maps, aspect="auto", cmap="magma", vmin=0, vmax=1, extent=(0, TIME_LENGTH - 1, len(class_names) - 0.5, -0.5))
        axis.set(title=method_name, yticks=np.arange(len(class_names)), yticklabels=class_names, ylabel="true dose")
    axes[-1].set_xlabel("time index")
    figure.colorbar(image, ax=axes, label="normalized attribution", shrink=0.8)
    figure.suptitle(f"Per-dose attribution maps — {experiment_name}")
    figure.subplots_adjust(top=0.90, right=0.88, hspace=0.32)

<a id="examples"></a>
## 4. Representative trajectories

One deterministic trajectory per true dose is shown for each model. Point color represents normalized relevance. These plots are examples rather than evidence of population-level behavior; interpret them together with the dose-average maps.

In [ ]:
def plot_colored_trajectory(axis, trajectory, attribution, title):
    axis.plot(time_axis, trajectory, color="0.45", linewidth=1.0, zorder=1)
    points = axis.scatter(time_axis, trajectory, c=attribution, cmap="magma", vmin=0, vmax=1, s=10, zorder=2)
    axis.set_title(title, fontsize=9)
    axis.grid(alpha=0.15)
    return points
representative_positions = [
    int(torch.nonzero(analysis_targets == class_index, as_tuple=False)[0])
    for class_index in range(len(class_names))
]

In [ ]:
for experiment_name, methods in attributions.items():
    figure, axes = plt.subplots(len(class_names), len(methods), figsize=(15, 2.2 * len(class_names)), sharex=True)
    for row, position in enumerate(representative_positions):
        trajectory = analysis_inputs[position, 0].detach().cpu().numpy()
        for column, (method_name, values) in enumerate(methods.items()):
            plot_colored_trajectory(
                axes[row, column], trajectory, values[position, 0].cpu().numpy(),
                f"{class_names[row]} — {method_name}",
            )
    for axis in axes[-1]: axis.set_xlabel("time index")
    figure.suptitle(f"Representative tune trajectories — {experiment_name}")
    figure.tight_layout()

<a id="spectral"></a>
## 5. Spectral Grad-CAM

Only models with an FFT branch appear below. The horizontal axis is normalized frequency. Magnitude-only FFT features discard phase, so their Grad-CAM cannot identify when an event occurred. Input-level Integrated Gradients and Temporal Occlusion still provide temporal views because they differentiate or perturb the original trajectory through the complete model.

In [ ]:
frequency_axis = np.fft.rfftfreq(TIME_LENGTH)
fft_plotted = False
for experiment_name, maps in spectral_cams.items():
    for layer_name, values in maps.items():
        fft_plotted = True
        figure, axis = plt.subplots(figsize=(10, 4))
        for class_index, class_name in enumerate(class_names):
            mean_map = values[analysis_targets == class_index].mean(dim=0).squeeze().cpu().numpy()
            axis.plot(frequency_axis, mean_map, label=class_name)
        axis.set(title=f"Spectral Grad-CAM — {experiment_name}: {layer_name}", xlabel="normalized frequency", ylabel="normalized attribution")
        axis.legend(ncol=3); axis.grid(alpha=0.2); figure.tight_layout()
if not fft_plotted: print("None of the selected top-three models contains an FFT branch.")

<a id="validation"></a>
## 6. Agreement and deletion tests

Spearman correlation compares the ranking of time points, not their absolute scale. The deletion test then masks the most important 17-point window and compares its true-class logit drop with a seed-matched random window. A useful explanation should usually produce a larger top-window drop than the random control. This is a faithfulness diagnostic, not proof that the same interval is biologically causal.

In [ ]:
agreement_rows, deletion_rows = [], []
for experiment_name, methods in attributions.items():
    method_items = list(methods.items())
    for left_index, (left_name, left_values) in enumerate(method_items):
        for right_name, right_values in method_items[left_index + 1:]:
            correlations = [
                spearmanr(left_values[index].cpu().flatten(), right_values[index].cpu().flatten()).statistic
                for index in range(len(analysis_inputs))
]
            agreement_rows.append({
                "experiment": experiment_name, "comparison": f"{left_name} vs {right_name}",
                "mean_spearman": np.nanmean(correlations), "std_spearman": np.nanstd(correlations),
            })
    for method_name in ("Integrated Gradients", "Layer Grad-CAM"):
        deletion = temporal_window_deletion(
            models[experiment_name], analysis_inputs, analysis_targets, methods[method_name],
            window_size=17, random_seed=42,
        )
        for strategy in ("top", "random"):
            values = deletion[strategy].cpu().numpy()
            deletion_rows.append({
                "experiment": experiment_name, "method": method_name, "window": strategy,
                "mean_logit_drop": values.mean(), "std_logit_drop": values.std(),
            })
agreement = pd.DataFrame(agreement_rows)
deletion_summary = pd.DataFrame(deletion_rows)
display(agreement.round(3))
display(deletion_summary.round(3))

In [ ]:
deletion_plot = deletion_summary.pivot(index=["experiment", "method"], columns="window", values="mean_logit_drop")
deletion_plot[["top", "random"]].plot.bar(figsize=(12, 5), rot=35)
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("true-class logit drop"); plt.title("Attribution-guided versus random window deletion")
plt.grid(axis="y", alpha=0.2); plt.tight_layout()

<a id="interpretation"></a>
## 7. Interpretation checklist

A strong result is not merely a visually sharp heatmap. Look for regions that recur within a dose, remain similar across Integrated Gradients and Grad-CAM, and produce a larger logit drop than random windows. Then relate those regions cautiously to early activation, rise, peak, decay or late plateau.

Important limitations:

- Selection and explanation both use `validate`; explanations describe the frozen candidates but are not an independent estimate.
- Attribution magnitude is model- and method-specific; normalization removes absolute effect size.
- A zero baseline is interpretable after global z-scoring but not uniquely correct.
- Grad-CAM is coarse and layer-dependent.
- FFT magnitude relevance is spectral, not temporally localized.
- Correlated neighboring time points permit multiple equally plausible explanations.

Next checks should repeat explanations across training seeds, compare alternative baselines and window widths, quantify attribution stability under small temporal shifts, and use biologically defined phase masks. None of those checks should trigger model reselection using test.